In [10]:
import numpy as np
import pandas as pd

In [11]:

# # 1. Cargar la base original

df_xls = pd.read_excel("/content/drive/MyDrive/Curso introducción a Python para Economistas/pib_9paises_2020_2022.xlsx")



# df_colores_dist["desocupados"] = (
#     df_colores_dist["poblacion_economicamente_activa"] - df_colores_dist["ocupados"]
# ).round(0)

df_colores_dist.head()

,pais,pib_per_capita,densidad_poblacional
0,Amarillo,0.006038,45.534151
1,Azul,0.009458,23.837482
2,Rojo,0.013739,16.546763
3,Verde,0.016256,25.793651


In [12]:

# 2. Mapa de países -> colores (uno por cada país)
colores = [
    "Rojo", "Azul", "Verde",
    "Amarillo", "Naranja", "Morado",
    "Cafe", "Gris", "Blanco"
]

paises_originales = sorted(df_xls["pais"].unique())
mapa_pais_color = dict(zip(paises_originales, colores))

# 3. Crear copia y reemplazar nombres por colores
df_colores_dist = df_xls.copy()
df_colores_dist["pais"] = df_colores_dist["pais"].map(mapa_pais_color)

# 4. Distorsión 1–9 % (arriba o abajo) en area_km2, poblacion y PIB
np.random.seed(2025)  # para reproducibilidad

for col in ["area_km2", "poblacion", "PIB"]:
    n = len(df_colores_dist)
    # tamaño de la distorsión: entre 1 % y 9 %
    delta = np.random.uniform(0.01, 0.09, size=n)
    # signo aleatorio: + o -
    sign = np.random.choice([-1, 1], size=n)
    factor = 1 + sign * delta

    df_colores_dist[col] = (df_colores_dist[col] * factor).round(0)

# mantener poblacion.1 sincronizada con poblacion
if "poblacion.1" in df_colores_dist.columns:
    df_colores_dist["poblacion.1"] = df_colores_dist["poblacion"]

# 5. Recalcular variables derivadas básicas para que sean coherentes
# pib per cápita
df_colores_dist["pib_pc"] = (df_colores_dist["PIB"] / df_colores_dist["poblacion"]).round(0)

# PEA, ocupados y desocupados a partir de nuevas poblaciones y tasas
df_colores_dist["poblacion_economicamente_activa"] = (
    df_colores_dist["poblacion"] * (1 - df_colores_dist["tasa_inactividad"] / 100)
).round(0)

df_colores_dist["ocupados"] = (
    df_colores_dist["poblacion_economicamente_activa"] * (1 - df_colores_dist["tasa_desempleo"] / 100)
).round(0)

df_colores_dist["desocupados"] = (
    df_colores_dist["poblacion_economicamente_activa"] - df_colores_dist["ocupados"]
).round(0)

df_colores_dist.head()


,pais,anio,area_km2,poblacion,poblacion.1,tasa_desempleo,tasa_inactividad,tasa_empleo,poblacion_economicamente_activa,ocupados,desocupados,pib_pc,PIB,SUDAMERICA
0,Rojo,2020,764091.0,5515938.0,5515938.0,13.235278,19.564867,67.199855,4436752.0,3849536.0,587216.0,33307.0,1.837178e+11,True
1,Rojo,2021,843584.0,6247801.0,6247801.0,3.807315,23.372010,72.820675,4787564.0,4605286.0,182278.0,23094.0,1.442840e+11,True
2,Rojo,2022,846377.0,6353026.0,6353026.0,6.305398,17.510680,76.183922,5240568.0,4910129.0,330439.0,45258.0,2.875258e+11,True
3,Azul,2020,1990797.0,15228598.0,15228598.0,5.765577,23.342451,70.891971,11673870.0,11000804.0,673066.0,29385.0,4.474889e+11,True
4,Azul,2021,1825721.0,14348143.0,14348143.0,5.638589,22.293805,72.067606,11149396.0,10520727.0,628669.0,17440.0,2.502274e+11,True


In [13]:
nombre_archivo = 'DataFrame_Sesion03_estudiantes.xlsx'

# 1. Exportar el DataFrame a Excel
# index=False es importante para no incluir la columna numérica automática de pandas.
df_colores_dist.to_excel(nombre_archivo, index=False)